In [2]:
from transformers import AutoTokenizer 
import torch

# Fast Tokenizers

In [43]:
Bert_tokenizer = AutoTokenizer.from_pretrained('bert-base-cased')
Roberta_tokenizer = AutoTokenizer.from_pretrained('roberta-base')
example = ["My name is Dhairya.81s","And I'm AIML developer"]
Bert_encoding = Bert_tokenizer(example)
Roberta_encoding = Roberta_tokenizer(example)

In [44]:
print("-"*50)
print("*"*10+"Bert Result"+"*"*10)
print(f"Tokens : {Bert_encoding.tokens()}")
print(f"word_ids : {Bert_encoding.word_ids()}")
print("-"*50)
print("*"*10+"RoBerta Result"+"*"*10)
print(f"Tokens : {Roberta_encoding.tokens()}")
print(f"word_ids : {Roberta_encoding.word_ids()}")
print("-"*50)

--------------------------------------------------
**********Bert Result**********
Tokens : ['[CLS]', 'My', 'name', 'is', 'D', '##hai', '##rya', '.', '81', '##s', '[SEP]']
word_ids : [None, 0, 1, 2, 3, 3, 3, 4, 5, 5, None]
--------------------------------------------------
**********RoBerta Result**********
Tokens : ['<s>', 'My', 'Ġname', 'Ġis', 'ĠDh', 'air', 'ya', '.', '81', 's', '</s>']
word_ids : [None, 0, 1, 2, 3, 3, 3, 4, 5, 6, None]
--------------------------------------------------


# Token Classification pipeline

In [49]:
from transformers import pipeline 
token_classifier = pipeline('token-classification',aggregation_strategy="simple")


No model was supplied, defaulted to dbmdz/bert-large-cased-finetuned-conll03-english and revision 4c53496.
Using a pipeline without specifying a model name and revision in production is not recommended.


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

BertForTokenClassification LOAD REPORT from: dbmdz/bert-large-cased-finetuned-conll03-english
Key                      | Status     |  | 
-------------------------+------------+--+-
bert.pooler.dense.bias   | UNEXPECTED |  | 
bert.pooler.dense.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [50]:
token_classifier("My name is Dhairya and I work at AppstoneLab , Katargam")

[{'entity_group': 'PER',
  'score': np.float32(0.99897677),
  'word': 'Dhairya',
  'start': 11,
  'end': 18},
 {'entity_group': 'ORG',
  'score': np.float32(0.973808),
  'word': 'AppstoneLab',
  'start': 33,
  'end': 44},
 {'entity_group': 'LOC',
  'score': np.float32(0.7338248),
  'word': 'Katargam',
  'start': 47,
  'end': 55}]

*Creating token classification pipeline without using **pipeline function***

In [51]:
from transformers import AutoTokenizer, AutoModelForTokenClassification
model_checkpoint = "dbmdz/bert-large-cased-finetuned-conll03-english"
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)
model = AutoModelForTokenClassification.from_pretrained(model_checkpoint)

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

BertForTokenClassification LOAD REPORT from: dbmdz/bert-large-cased-finetuned-conll03-english
Key                      | Status     |  | 
-------------------------+------------+--+-
bert.pooler.dense.bias   | UNEXPECTED |  | 
bert.pooler.dense.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [85]:
example = "My name is Dhairya and Swayam work at AppstoneLab in Katargam."
inputs = tokenizer(example,return_tensors="pt")
outputs = model(**inputs)

In [86]:
print(outputs.logits.shape)

torch.Size([1, 24, 9])


In [87]:
probabilities = torch.nn.functional.softmax(outputs.logits,dim=-1)[0]
preds = outputs.logits.argmax(dim=-1)[0].tolist()
label_preds = [model.config.id2label[element] for element in preds]


In [88]:
print("-"*100)
print("Prediction : ",label_preds)
print("Word Ids : ",inputs.tokens())
print("Word Ids : ",inputs.word_ids())
print("-"*100)

----------------------------------------------------------------------------------------------------
Prediction :  ['O', 'O', 'O', 'O', 'I-PER', 'I-PER', 'I-PER', 'O', 'I-PER', 'I-PER', 'I-PER', 'O', 'O', 'I-ORG', 'I-ORG', 'I-ORG', 'I-ORG', 'I-ORG', 'O', 'I-LOC', 'I-LOC', 'I-LOC', 'O', 'O']
Word Ids :  ['[CLS]', 'My', 'name', 'is', 'D', '##hai', '##rya', 'and', 'S', '##way', '##am', 'work', 'at', 'A', '##pps', '##tone', '##L', '##ab', 'in', 'Kat', '##ar', '##gam', '.', '[SEP]']
Word Ids :  [None, 0, 1, 2, 3, 3, 3, 4, 5, 5, 5, 6, 7, 8, 8, 8, 8, 8, 9, 10, 10, 10, 11, None]
----------------------------------------------------------------------------------------------------


# Fast Tokenizer in Question Answering Pipeline

In [1]:
from transformers import AutoTokenizer, AutoModelForQuestionAnswering
model_checkpoint = "distilbert-base-cased-distilled-squad"
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)
model = AutoModelForQuestionAnswering.from_pretrained(model_checkpoint)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


config.json:   0%|          | 0.00/473 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/213k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/436k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/261M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/102 [00:00<?, ?it/s]

In [ ]:
question = "Who is the best captain for team India in Cricket ?"
context = "while we talk about team India, the team is very balanced and all the credit goes the captain Rohit Sharma, he changed the indent of playing and took team India to its peak."

In [92]:
input_tensor = tokenizer(question,context,return_tensors="pt")
outputs = model(**input_tensor)

**Start_logits** : contains start_index logit values for all the tokens in the sequence. \
**End_logits** : contains end_index logit values for all the tokens in the sequence.

In [95]:
start_logits = outputs.start_logits
end_logits = outputs.end_logits
print(start_logits.shape,end_logits.shape)

torch.Size([1, 53]) torch.Size([1, 53])


In [143]:
sequence_ids = input_tensor.sequence_ids()
mask = [i != 1 for i in sequence_ids]
mask[0] = False 
mask = torch.tensor(mask)[None]

start_logits[mask] = -10000
end_logits[mask] = -10000

start_proba = torch.nn.functional.softmax(start_logits,dim=-1)[0]
end_proba = torch.nn.functional.softmax(end_logits,dim=-1)[0]

In [180]:
inputs_with_offsets = tokenizer(question, context, return_offsets_mapping=True)
offsets = inputs_with_offsets["offset_mapping"]

start_idx = start_proba.argmax().item()
end_idx = end_proba.argmax().item()

start_char, _ = offsets[start_index]
_, end_char = offsets[end_index]

result = {
    "answer : " : context[start_char:end_char],
}
result

{'answer : ': 'Rohit Sharma'}

**Sometimes we may have start index value greater than end Index value.**

In [ ]:
scores = start_proba[:, None] * end_proba[None, :] # a matrix containing all start * end proba score
max_index = scores.argmax().item()
start_index = max_index // scores.shape[1] #basic formula for accessing row value
end_index = max_index % scores.shape[1] #basic formula for accessing column value
print(scores[start_index, end_index])

tensor(0.9881, grad_fn=<SelectBackward0>)


In [160]:
inputs_with_offsets = tokenizer(question, context, return_offsets_mapping=True)
offsets = inputs_with_offsets["offset_mapping"]

start_char, _ = offsets[start_index]
_, end_char = offsets[end_index]
answer = context[start_char:end_char]

In [184]:
result = {
    "answer": answer,
    "start": start_char,
    "end": end_char,
    "score": scores[start_index, end_index].item(),
}
print(result)

{'answer': 'Rohit Sharma', 'start': 94, 'end': 106, 'score': 0.9881393313407898}


**Long context and multiple question answer handling**

In [19]:
questions = "Who is captain of team India in cricket.?"
long_context = """Rohit Sharma’s role during the ICC World Cups has evolved from being a prolific century-maker into a transformative leader who prioritizes team impact over personal milestones. During the 2019 World Cup, he etched his name in history by becoming the first player to score five centuries in a single edition, finishing as the tournament's leading run-scorer. However, by the 2023 World Cup, as captain, Rohit shifted his approach to a "fearless" brand of cricket, intentionally attacking from the first ball to maximize the powerplay. This selfless strategy provided India with explosive starts—exemplified by his 597 runs at a massive strike rate of 125.94—allowing the middle order to play with more freedom. His leadership saw India achieve a historic 10-match winning streak, proving that he was willing to trade potential "daddy hundreds" for early-inning dominance. Even with this aggressive shift, Rohit remains the all-time leader for the most centuries in World Cup history.While his highest score in a World Cup match is a brilliant 140 against Pakistan in 2019, his all-time career peak is the stuff of cricketing folklore. On November 13, 2014, at Eden Gardens, Rohit decimated the Sri Lankan bowling attack to score 264 runs, the highest individual score in the history of One Day Internationals (ODIs). This monumental innings included 33 boundaries and 9 sixes, and he remains the only cricketer in history to have scored three double-centuries in the ODI format. From his early days as a talented middle-order prospect to his current status as one of the most successful white-ball captains, Rohit’s legacy is defined by his ability to dominate the game’s biggest stages with unmatched timing and power."""

In [20]:
input_tensor = tokenizer(
    questions,
    long_context,
    stride=128,
    max_length=384,
    padding="longest",
    truncation="only_second",
    return_overflowing_tokens=True,
    return_offsets_mapping=True,
)

In [21]:
_ = input_tensor.pop('overflow_to_sample_mapping')
offsets = input_tensor.pop('offset_mapping')

In [22]:
input_tensor = input_tensor.convert_to_tensors('pt')
print(input_tensor['input_ids'].shape)

torch.Size([1, 381])


In [23]:
outputs = model(**input_tensor)
start_logits = outputs.start_logits
end_logits = outputs.end_logits
print(start_logits.shape,end_logits.shape)

torch.Size([1, 381]) torch.Size([1, 381])


In [24]:
sequence_ids = input_tensor.sequence_ids()
mask = [i!=1 for i in sequence_ids]
mask[0] = False 
mask = torch.logical_or(torch.tensor(mask)[None],(input_tensor['attention_mask']==0))
start_logits[mask] = -10000
end_logits[mask] = -10000

In [25]:
start_proba = torch.nn.functional.softmax(start_logits,dim=-1)
end_proba = torch.nn.functional.softmax(end_logits,dim=-1)

In [26]:
candidates = []
for sp,ep in zip(start_proba,end_proba):
    scores = sp[:,None] * ep[None,:]
    idx = scores.argmax().item()
    start_idx = idx // scores.shape[1]
    end_idx = idx % scores.shape[1]
    score = scores[start_idx,end_idx].item()
    candidates.append((start_idx,end_idx,score))

In [27]:
print(candidates)

[(12, 15, 0.6152747869491577)]


In [28]:
inputs_with_offsets = tokenizer(questions,long_context,return_offsets_mapping=True)
offsets = inputs_with_offsets['offset_mapping']

for candidate, offset in zip(candidates,offsets):
    start_token,end_token, score = candidate
    start_char,_ = offsets[start_token]
    _,end_char = offsets[end_token]
    results = {
        "answer" : long_context[start_char:end_char],
        "score" : score
    }
    print(results)


{'answer': 'Rohit Sharma', 'score': 0.6152747869491577}


# Byte-Pair Encoding tokenization

In [59]:
corpus = [
    "this file is for trial purpose.",
    "this file contains classified information.",
    "info is key to data",
    "hugging face has the best interface for learning",
    "cricket isnt only game played in India"
]

In [60]:
from collections import defaultdict 

word_freqs = defaultdict(int)
for text in corpus:
    words_with_offset = tokenizer.backend_tokenizer.pre_tokenizer.pre_tokenize_str(text)
    new_words = [word for word , offser in words_with_offset]
    for word in new_words:
        word_freqs[word] += 1 
print(word_freqs)

defaultdict(<class 'int'>, {'this': 2, 'file': 2, 'is': 2, 'for': 2, 'trial': 1, 'purpose': 1, '.': 2, 'contains': 1, 'classified': 1, 'information': 1, 'info': 1, 'key': 1, 'to': 1, 'data': 1, 'hugging': 1, 'face': 1, 'has': 1, 'the': 1, 'best': 1, 'interface': 1, 'learning': 1, 'cricket': 1, 'isnt': 1, 'only': 1, 'game': 1, 'played': 1, 'in': 1, 'India': 1})


In [61]:
alphabet = []

for word in word_freqs.keys():
    for letter in word:
        if letter not in alphabet:
            alphabet.append(letter)
alphabet.sort()

print(alphabet)

['.', 'I', 'a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'k', 'l', 'm', 'n', 'o', 'p', 'r', 's', 't', 'u', 'y']


In [62]:
vocab = ["<|endoftext|>"] + alphabet.copy()

In [63]:
splits = {word: [c for c in word] for word in word_freqs.keys()}
splits

{'this': ['t', 'h', 'i', 's'],
 'file': ['f', 'i', 'l', 'e'],
 'is': ['i', 's'],
 'for': ['f', 'o', 'r'],
 'trial': ['t', 'r', 'i', 'a', 'l'],
 'purpose': ['p', 'u', 'r', 'p', 'o', 's', 'e'],
 '.': ['.'],
 'contains': ['c', 'o', 'n', 't', 'a', 'i', 'n', 's'],
 'classified': ['c', 'l', 'a', 's', 's', 'i', 'f', 'i', 'e', 'd'],
 'information': ['i', 'n', 'f', 'o', 'r', 'm', 'a', 't', 'i', 'o', 'n'],
 'info': ['i', 'n', 'f', 'o'],
 'key': ['k', 'e', 'y'],
 'to': ['t', 'o'],
 'data': ['d', 'a', 't', 'a'],
 'hugging': ['h', 'u', 'g', 'g', 'i', 'n', 'g'],
 'face': ['f', 'a', 'c', 'e'],
 'has': ['h', 'a', 's'],
 'the': ['t', 'h', 'e'],
 'best': ['b', 'e', 's', 't'],
 'interface': ['i', 'n', 't', 'e', 'r', 'f', 'a', 'c', 'e'],
 'learning': ['l', 'e', 'a', 'r', 'n', 'i', 'n', 'g'],
 'cricket': ['c', 'r', 'i', 'c', 'k', 'e', 't'],
 'isnt': ['i', 's', 'n', 't'],
 'only': ['o', 'n', 'l', 'y'],
 'game': ['g', 'a', 'm', 'e'],
 'played': ['p', 'l', 'a', 'y', 'e', 'd'],
 'in': ['i', 'n'],
 'India': ['I

In [64]:
def compute_pair_freqs(splits):
    pair_freqs = defaultdict(int)
    for word, freq in word_freqs.items():
        split = splits[word]
        if len(split) == 1:
            continue
        for i in range(len(split) - 1):
            pair = (split[i], split[i + 1])
            pair_freqs[pair] += freq
    return pair_freqs

In [65]:
pair_freqs = compute_pair_freqs(splits)
pair_freqs

defaultdict(int,
            {('t', 'h'): 3,
             ('h', 'i'): 2,
             ('i', 's'): 5,
             ('f', 'i'): 3,
             ('i', 'l'): 2,
             ('l', 'e'): 3,
             ('f', 'o'): 4,
             ('o', 'r'): 3,
             ('t', 'r'): 1,
             ('r', 'i'): 2,
             ('i', 'a'): 2,
             ('a', 'l'): 1,
             ('p', 'u'): 1,
             ('u', 'r'): 1,
             ('r', 'p'): 1,
             ('p', 'o'): 1,
             ('o', 's'): 1,
             ('s', 'e'): 1,
             ('c', 'o'): 1,
             ('o', 'n'): 3,
             ('n', 't'): 3,
             ('t', 'a'): 2,
             ('a', 'i'): 1,
             ('i', 'n'): 7,
             ('n', 's'): 1,
             ('c', 'l'): 1,
             ('l', 'a'): 2,
             ('a', 's'): 2,
             ('s', 's'): 1,
             ('s', 'i'): 1,
             ('i', 'f'): 1,
             ('i', 'e'): 1,
             ('e', 'd'): 2,
             ('n', 'f'): 2,
             ('r', 'm'): 1,
   

In [66]:
best_pair = ""
max_freq = None

for pair, freq in pair_freqs.items():
    if max_freq is None or max_freq < freq:
        best_pair = pair
        max_freq = freq

print(best_pair, max_freq)

('i', 'n') 7


In [67]:
def merge_pair(a, b, splits):
    for word in word_freqs:
        split = splits[word]
        if len(split) == 1:
            continue

        i = 0
        while i < len(split) - 1:
            if split[i] == a and split[i + 1] == b:
                split = split[:i] + [a + b] + split[i + 2 :]
            else:
                i += 1
        splits[word] = split
    return splits

In [69]:
merges = {('i','n'):"in"}
vocab.append('in')

In [70]:
splits = merge_pair("i", "n", splits)
print(splits["information"])

['in', 'f', 'o', 'r', 'm', 'a', 't', 'i', 'o', 'n']


In [71]:
vocab_size = 50

while len(vocab) < vocab_size:
    pair_freqs = compute_pair_freqs(splits)
    best_pair = ""
    max_freq = None
    for pair, freq in pair_freqs.items():
        if max_freq is None or max_freq < freq:
            best_pair = pair
            max_freq = freq
    splits = merge_pair(*best_pair, splits)
    merges[best_pair] = best_pair[0] + best_pair[1]
    vocab.append(best_pair[0] + best_pair[1])

In [79]:
print(merges)

{('i', 'n'): 'in', ('i', 's'): 'is', ('f', 'o'): 'fo', ('t', 'h'): 'th', ('f', 'i'): 'fi', ('l', 'e'): 'le', ('fo', 'r'): 'for', ('o', 'n'): 'on', ('th', 'is'): 'this', ('fi', 'le'): 'file', ('r', 'i'): 'ri', ('t', 'a'): 'ta', ('l', 'a'): 'la', ('e', 'd'): 'ed', ('k', 'e'): 'ke', ('in', 'g'): 'ing', ('f', 'a'): 'fa', ('fa', 'c'): 'fac', ('fac', 'e'): 'face', ('t', 'ri'): 'tri', ('tri', 'a'): 'tria', ('tria', 'l'): 'trial', ('p', 'u'): 'pu', ('pu', 'r'): 'pur', ('pur', 'p'): 'purp', ('purp', 'o'): 'purpo'}


# WordPiece Tokenization

In [80]:
corpus = [
    "this file is for trial purpose.",
    "this file contains classified information.",
    "info is key to data",
    "hugging face has the best interface for learning",
    "cricket isnt only game played in India"
]

In [81]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("bert-base-cased")

In [82]:
from collections import defaultdict

word_freqs = defaultdict(int)
for text in corpus:
    words_with_offsets = tokenizer.backend_tokenizer.pre_tokenizer.pre_tokenize_str(text)
    new_words = [word for word, offset in words_with_offsets]
    for word in new_words:
        word_freqs[word] += 1

word_freqs

defaultdict(int,
            {'this': 2,
             'file': 2,
             'is': 2,
             'for': 2,
             'trial': 1,
             'purpose': 1,
             '.': 2,
             'contains': 1,
             'classified': 1,
             'information': 1,
             'info': 1,
             'key': 1,
             'to': 1,
             'data': 1,
             'hugging': 1,
             'face': 1,
             'has': 1,
             'the': 1,
             'best': 1,
             'interface': 1,
             'learning': 1,
             'cricket': 1,
             'isnt': 1,
             'only': 1,
             'game': 1,
             'played': 1,
             'in': 1,
             'India': 1})

In [83]:
alphabet = []
for word in word_freqs.keys():
    if word[0] not in alphabet:
        alphabet.append(word[0])
    for letter in word[1:]:
        if f"##{letter}" not in alphabet:
            alphabet.append(f"##{letter}")

alphabet.sort()
alphabet

print(alphabet)

['##a', '##c', '##d', '##e', '##f', '##g', '##h', '##i', '##k', '##l', '##m', '##n', '##o', '##p', '##r', '##s', '##t', '##u', '##y', '.', 'I', 'b', 'c', 'd', 'f', 'g', 'h', 'i', 'k', 'l', 'o', 'p', 't']


In [84]:
vocab = ["[PAD]", "[UNK]", "[CLS]", "[SEP]", "[MASK]"] + alphabet.copy()

In [85]:
splits = {
    word: [c if i == 0 else f"##{c}" for i, c in enumerate(word)]
    for word in word_freqs.keys()
}

In [91]:
splits

{'this': ['t', '##h', '##i', '##s'],
 'file': ['f', '##i', '##l', '##e'],
 'is': ['i', '##s'],
 'for': ['f', '##o', '##r'],
 'trial': ['t', '##r', '##i', '##a', '##l'],
 'purpose': ['p', '##u', '##r', '##p', '##o', '##s', '##e'],
 '.': ['.'],
 'contains': ['c', '##o', '##n', '##t', '##a', '##i', '##n', '##s'],
 'classified': ['c',
  '##l',
  '##a',
  '##s',
  '##s',
  '##i',
  '##f',
  '##i',
  '##e',
  '##d'],
 'information': ['i',
  '##n',
  '##f',
  '##o',
  '##r',
  '##m',
  '##a',
  '##t',
  '##i',
  '##o',
  '##n'],
 'info': ['i', '##n', '##f', '##o'],
 'key': ['k', '##e', '##y'],
 'to': ['t', '##o'],
 'data': ['d', '##a', '##t', '##a'],
 'hugging': ['h', '##u', '##g', '##g', '##i', '##n', '##g'],
 'face': ['f', '##a', '##c', '##e'],
 'has': ['h', '##a', '##s'],
 'the': ['t', '##h', '##e'],
 'best': ['b', '##e', '##s', '##t'],
 'interface': ['i', '##n', '##t', '##e', '##r', '##f', '##a', '##c', '##e'],
 'learning': ['l', '##e', '##a', '##r', '##n', '##i', '##n', '##g'],
 'cricket

In [86]:
def compute_pair_scores(splits):
    letter_freqs = defaultdict(int)
    pair_freqs = defaultdict(int)
    for word, freq in word_freqs.items():
        split = splits[word]
        if len(split) == 1:
            letter_freqs[split[0]] += freq
            continue
        for i in range(len(split) - 1):
            pair = (split[i], split[i + 1])
            letter_freqs[split[i]] += freq
            pair_freqs[pair] += freq
        letter_freqs[split[-1]] += freq

    scores = {
        pair: freq / (letter_freqs[pair[0]] * letter_freqs[pair[1]])
        for pair, freq in pair_freqs.items()
    }
    return scores

In [87]:
pair_scores = compute_pair_scores(splits)
for i, key in enumerate(pair_scores.keys()):
    print(f"{key}: {pair_scores[key]}")
    if i >= 5:
        break


('t', '##h'): 0.2
('##h', '##i'): 0.05128205128205128
('##i', '##s'): 0.013986013986013986
('f', '##i'): 0.03076923076923077
('##i', '##l'): 0.02564102564102564
('##l', '##e'): 0.023809523809523808


In [88]:
best_pair = ""
max_score = None
for pair, score in pair_scores.items():
    if max_score is None or max_score < score:
        best_pair = pair
        max_score = score

print(best_pair, max_score)

('##c', '##k') 0.3333333333333333


In [89]:
def merge_pair(a, b, splits):
    for word in word_freqs:
        split = splits[word]
        if len(split) == 1:
            continue
        i = 0
        while i < len(split) - 1:
            if split[i] == a and split[i + 1] == b:
                merge = a + b[2:] if b.startswith("##") else a + b
                split = split[:i] + [merge] + split[i + 2 :]
            else:
                i += 1
        splits[word] = split
    return splits

In [92]:
splits = merge_pair("##c", "##k", splits)
splits["cricket"]

['c', '##r', '##i', '##ck', '##e', '##t']

In [93]:
vocab_size = 70
while len(vocab) < vocab_size:
    scores = compute_pair_scores(splits)
    best_pair, max_score = "", None
    for pair, score in scores.items():
        if max_score is None or max_score < score:
            best_pair = pair
            max_score = score
    splits = merge_pair(*best_pair, splits)
    new_token = (
        best_pair[0] + best_pair[1][2:]
        if best_pair[1].startswith("##")
        else best_pair[0] + best_pair[1]
    )
    vocab.append(new_token)

In [94]:
print(vocab)

['[PAD]', '[UNK]', '[CLS]', '[SEP]', '[MASK]', '##a', '##c', '##d', '##e', '##f', '##g', '##h', '##i', '##k', '##l', '##m', '##n', '##o', '##p', '##r', '##s', '##t', '##u', '##y', '.', 'I', 'b', 'c', 'd', 'f', 'g', 'h', 'i', 'k', 'l', 'o', 'p', 't', 'pu', 'hu', 'hug', 'hugg', 'th', 'pl', 'pur', 'purp', 'purpo', 'purpos', 'da', 'dat', 'data', '##ac', '##fac', 'fac', '##rfac', 'ha', 'ga', 'gam', '##rm', '##orm', '##form', '##forma', '##format', 'pla', 'play', 'tr', 'to', 'fo', 'for', '##fo']


# UniGram tokenization